In [1]:
# ====================================================================
# Word Embeddings: Illustration of Embedding Training
# ====================================================================

import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# ── 1. Corpus ─────────────────────────────────────────────────────────────────
# A corpus is just a collection of sentences (our "training data").
# Each sentence follows the same pattern: [animal] [verb] [size] [colour] [prey]
# The model will learn word relationships purely from these co-occurrence patterns.

# corpus = [
#     "CAT chases small white mouse",
#     "dog chases big black mouse",
#     "bird chases tiny blue fly",
#     "cat hunts small grey mouse",
#     "dog hunts big brown mouse",
#     "bird hunts tiny green fly", 
# ]

# corpus = [
#     "haoming likes leo tea",
#     "haoming wants scorpio coffee",
#     "haoming orders gemini juice",
#     "jianzuo likes virgo coffee",
#     "jianzuo buys aries water",
#     "jianzuo loves pisces tea",
#     "xiaojun orders libra coke",
#     "xiaojun likes scorpio water",
#     "xiaojun wants pisces juice",
#     "huishan loves leo juice",
#     "huishan orders aries tea",
#     "huishan buys gemini coffee",
#     "shihnyien wants virgo water",
#     "shihnyien likes gemini coke",
#     "shihnyien orders pisces coffee",
#     "samuel buys scorpio tea",
#     "samuel loves libra juice",
#     "samuel wants aries coke",
#     "kuibin orders leo water",
#     "kuibin likes pisces coke",
#     "kuibin buys libra tea",
#     "yanwen loves gemini coffee",
#     "yanwen orders virgo juice",
#     "yanwen wants leo water",
#     "natalie buys aries tea",
#     "natalie likes libra coffee",
#     "natalie loves scorpio coke",
#     "fangting wants pisces juice",
#     "fangting buys leo water",
#     "fangting orders gemini tea",
#     "cynthia loves virgo water",
#     "cynthia wants libra tea",
#     "cynthia buys scorpio juice",
#     "szevy orders aries coke",
#     "szevy likes gemini water",
#     "szevy loves leo coffee",
#     "viknesh buys pisces coffee",
#     "viknesh orders libra water",
#     "viknesh wants virgo tea",
#     "david loves aries juice",
#     "david buys gemini coke",
#     "david likes scorpio water",
# ]

corpus = [
    "money at bank",
    "cash at bank",
    "funds at bank",
    "dollars at bank",
    "money at vault",
    "cash at vault",
    "funds at vault",
    "dollars at vault",

    "fish near bank",
    "river near bank",
    "trees near bank",
    "grass near bank",
    "fish near shore",
    "river near shore",
    "trees near shore",
    "grass near shore",
]



In [3]:
# Words to check — one from each category so we can see if the model
# groups them correctly (animals together, sizes together, etc.)

# words = ["cat", "dog", "bird", "small", "big", "tiny", "white", "black", "blue", "mouse", "fly"]

# words = ["kuibin", "haoming", "samuel", "natalie", "david", "viknesh",
#          "likes", "orders", "loves", "buys", "wants",
#          "aries", "leo", "virgo", "pisces",
#          "tea", "coffee", "coke", "water", "juice"]

# clusters to expect in 3D:

#   Names (14): haoming, jianzuo, xiaojun, huishan, shihnyien, samuel, kuibin, yanwen, natalie, fangting, cynthia, szevy, viknesh, david
#   Verbs (5): likes, wants, orders, buys, loves
#   Signs (7): leo, scorpio, gemini, virgo, aries, pisces, libra
#   Drinks (5): tea, coffee, juice, coke, water


words = ["money", "cash", "funds", "dollars", "vault",
         "fish", "river", "trees", "grass", "shore",
         "bank"]



# The problem: Word2Vec gives each word exactly ONE vector. So "bank" gets a single point in 3D space — it can't be in two places at once. It would end up somewhere in between the two meanings, belonging to neither cluster cleanly.


#     money-bank words ●  ●  ●
#                               ● bank   ← stuck in the middle
#     river-bank words ●  ●  ●

# This is exactly why contextual embeddings (BERT, GPT) were invented — they give "bank" a different vector depending on the surrounding words. Our Word2Vec model can't do that.

In [4]:
# ── 2. Vocabulary ─────────────────────────────────────────────────────────────
# A vocabulary maps every unique word to a unique integer ID.
# This is the simplest possible tokenizer: split on spaces, assign IDs.
#
# Why do we need this?
#   Neural networks can't read text — they only work with numbers.
#   So we convert:  "cat" → 0,  "chases" → 1,  "small" → 2, ...
#
# How it works:
#   - Walk through every word in every sentence
#   - First time we see a word → give it the next available ID (len(vocab))
#   - Already seen it → skip (the `if word not in vocab` check)
#
# Example trace for first sentence "cat chases small white mouse":
#   "cat"    not in {} → vocab = {"cat": 0}
#   "chases" not in {"cat":0} → vocab = {"cat": 0, "chases": 1}
#   "small"  not in ... → vocab = {"cat": 0, "chases": 1, "small": 2}
#   "white"  → vocab = {..., "white": 3}
#   "mouse"  → vocab = {..., "mouse": 4}
#   (continues with second sentence: "dog"→5, "big"→6, "black"→7, ...)
#   ("chases" and "mouse" already in vocab → skipped)
def build_vocab(corpus):
    vocab = {}
    for sent in corpus:
        for word in sent.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(corpus)
# vocab is now something like:
#   {"CAT": 0, "chases": 1, "small": 2, "white": 3, "mouse": 4,
#    "dog": 5, "big": 6, "black": 7, "bird": 8, "tiny": 9, "blue": 10,
#    "fly": 11, "cat": 12, "hunts": 13, "grey": 14, "brown": 15, "green": 16}
#
# Note: "CAT" and "cat" get separate IDs because this tokenizer is case-sensitive.
#       A real tokenizer would typically lowercase everything first.

# Reverse lookup: given an ID, get back the word
# e.g. idx2w[0] → "CAT", idx2w[4] → "mouse"
idx2w = {i: w for w, i in vocab.items()}

# ── 3a. CBOW pairs ────────────────────────────────────────────────────────────
# All context words together → one center word
# "cat chases [small] white mouse"
# [cat, chases, white, mouse] → small
# Output: ([list of context ids], center) pairs
#
# CBOW = Continuous Bag of Words
# The idea: if you see the surrounding words, can you predict the middle word?
#
# With window=2, for each word in a sentence, we look 2 words left and 2 words right.
# The surrounding words are "context", the middle word is the "center" (target).
#
# Example: "cat chases [small] white mouse"  (window=2, center=small)
#   2 left:  cat, chases
#   2 right: white, mouse
#   context = [cat, chases, white, mouse] → predict center = small
#
# Edge cases: words near the start/end of a sentence have fewer context words
#   "[cat] chases small white mouse"  (center=cat, position 0)
#   0 left (nothing before it), 2 right: chases, small
#   context = [chases, small] → predict center = cat   (only 2 context words)
def get_cbow_pairs(corpus, vocab, window=2):
    pairs = []
    for sent in corpus:
        ids = [vocab[w] for w in sent.split()]
        for i, center in enumerate(ids):
            context = []
            for j in range(max(0, i-window), min(len(ids), i+window+1)):
                if i != j:
                    context.append(ids[j])           # collect all context words
            pairs.append((context, center))          # (many, one)
    return pairs

# ── 3b. Skip-gram pairs ───────────────────────────────────────────────────────
# One center word → each context word separately
# "cat chases [small] white mouse"
# small→cat, small→chases, small→white, small→mouse
# Output: many (center, context) pairs
#
# Skip-gram is the reverse of CBOW:
# Given the center word, can you predict each surrounding word?
#
# Same window, but instead of one training pair per position,
# we create a SEPARATE pair for each context word:
#   center=small → (small, cat), (small, chases), (small, white), (small, mouse)
#   That's 4 pairs from one position! (vs. 1 pair in CBOW)
#
# This is why skip-gram produces more pairs than CBOW (84 vs 30 in our corpus).
def get_skipgram_pairs(corpus, vocab, window=2):
    pairs = []
    for sent in corpus:
        ids = [vocab[w] for w in sent.split()]
        for i, center in enumerate(ids):
            for j in range(max(0, i-window), min(len(ids), i+window+1)):
                if i != j:
                    pairs.append((center, ids[j]))   # (one, one)
    return pairs

# ── 3c. Generate pairs and show examples ─────────────────────────────────────────
# Now we actually call both functions to create all the training pairs.
# These are the pairs the models will learn from in the training step later.

cbow_pairs     = get_cbow_pairs(corpus, vocab)
skipgram_pairs = get_skipgram_pairs(corpus, vocab)

# ── Print CBOW pairs ─────────────────────────────────────────────────────────
# Each row shows: [list of context words] → center word
# e.g.  ['chases', 'small']  →  CAT
# means: given the context words "chases" and "small", the target is "CAT"

col_width = 40

print(f"Sentences:")
print(" ")
for i in range(len(corpus)):
     print(f"       {corpus[i]}")

print(" ")
print(f"CBOW pairs:      {len(cbow_pairs)}")

print(f"{'Context':<{col_width}}   {'Center'}")
print("-" * (col_width + 12))

for context, center in cbow_pairs:
    context_words = [idx2w[idx] for idx in context]  # convert IDs back to words for display
    center_word   = idx2w[center]
    print(f"{str(context_words):<{col_width}}   {center_word}")

# ── Print Skip-gram pairs ────────────────────────────────────────────────────
# Each row shows: center word → context word
# e.g.  CAT  →  chases
# means: given the center word "CAT", one of its context words is "chases"
# Notice there are many more rows (84 vs 30) because each center-context
# combination is its own separate pair.

print(f"\nSkip-gram pairs: {len(skipgram_pairs)}")
print(f"{'Center':<{col_width}}   {'Context'}")   # ← headers swapped
print("-" * (col_width + 12))
for center, context in skipgram_pairs:            # ← variables swapped
    center_word  = idx2w[center]
    context_word = idx2w[context]                 # ← single word, not a list
    print(f"{center_word:<{col_width}}   {context_word}")

Sentences:
 
       money at bank
       cash at bank
       funds at bank
       dollars at bank
       money at vault
       cash at vault
       funds at vault
       dollars at vault
       fish near bank
       river near bank
       trees near bank
       grass near bank
       fish near shore
       river near shore
       trees near shore
       grass near shore
 
CBOW pairs:      48
Context                                    Center
----------------------------------------------------
['at', 'bank']                             money
['money', 'bank']                          at
['money', 'at']                            bank
['at', 'bank']                             cash
['cash', 'bank']                           at
['cash', 'at']                             bank
['at', 'bank']                             funds
['funds', 'bank']                          at
['funds', 'at']                            bank
['at', 'bank']                             dollars
['dollars', 'bank']    

In [5]:
# ── 4a. CBOW Model ────────────────────────────────────────────────────────────
#
# What is an embedding?
#   An embedding is a lookup table that maps each word ID to a vector of numbers.
#   e.g. "cat" (id=0) → [0.3, -0.1, 0.8, ...]   (a list of 8 numbers if embed_dim=8)
#
#   These numbers start random and are learned during training.
#   After training, words that appear in similar contexts will have similar vectors.
#   That's the whole point — we turn words into numbers that capture meaning.
#
# Why TWO embedding tables (self.context and self.center)?
#   Each word plays two different roles in our training pairs:
#     - Sometimes it's a CONTEXT word (the surrounding words)
#     - Sometimes it's the CENTER word (the word being predicted)
#
#   We give each role its own lookup table so the model can learn these roles
#   independently. (In practice, after training we typically only use one of them —
#   usually center — as the final word embedding.)
#
# What does the model actually compute?
#   A single number (score) that says: "how well does this context match this center?"
#   High score → these words likely belong together
#   Low score  → these words probably don't belong together
#   The score is computed via dot product (multiply + sum), which measures
#   how similar two vectors are pointing in the same direction.

class CBOW(nn.Module):

    def __init__(self, vocab_size, embed_dim):
        # Called ONCE when you write: cbow_model = CBOW(V, embed_dim=8)
        # Builds two empty embedding tables, e.g. vocab_size=12, embed_dim=8:
        #
        #   self.context → table shape [12 x 8]  (12 words, 8 numbers each)
        #   self.center  → table shape [12 x 8]
        #
        # Both filled with random numbers at first
        super().__init__()
        self.context = nn.Embedding(vocab_size, embed_dim)
        self.center  = nn.Embedding(vocab_size, embed_dim)

    def forward(self, cb_context, cb_center):
        # Called EVERY epoch when you write: cbow_model(cb_context, cb_center)
        # PyTorch automatically routes them here via __call__
        #
        # Example values flowing through (embed_dim=4 for simplicity):
        #
        # cb_context = [[3, 4, 7, 10]]       ← context ids for "small"
        #                                       [chases, small, white, mouse]
        #
        # cb_center  = [2]                   ← center id for "small"

        ct = self.context(cb_context)
        # Looks up each context word's vector:
        #   chases → [ 0.5, -0.1,  0.3,  0.8]
        #   small  → [-0.2,  0.4,  0.7, -0.3]
        #   white  → [ 0.1,  0.9, -0.4,  0.2]
        #   mouse  → [ 0.6, -0.5,  0.2,  0.4]
        #   ct shape: (1, 4, 4) → (batch=1, 4 context words, 4 dims)

        ct = ct.mean(dim=1)
        # Average all 4 context vectors into one:
        #   [ (0.5-0.2+0.1+0.6)/4,
        #     (-0.1+0.4+0.9-0.5)/4,
        #     (0.3+0.7-0.4+0.2)/4,
        #     (0.8-0.3+0.2+0.4)/4 ]
        #   = [0.25, 0.18, 0.20, 0.28]
        #   ct shape: (1, 4) → one averaged vector

        c  = self.center(cb_center)
        # Looks up center word vector:
        #   small → [0.3, 0.2, 0.4, 0.1]
        #   c shape: (1, 4)

        return (ct * c).sum(dim=1)
        # Element-wise multiply then sum = dot product:
        #   ct      = [0.25, 0.18, 0.20, 0.28]
        #   c       = [0.30, 0.20, 0.40, 0.10]
        #   ct * c  = [0.075, 0.036, 0.080, 0.028]
        #   sum     = 0.219  ← final score for this pair


# ── 4b. Skip-gram Model ───────────────────────────────────────────────────────
#
# Skip-gram has the exact same structure as CBOW (two embedding tables + dot product)
# but the data flows differently:
#
#   CBOW:      [many context words] → average → dot product with center → score
#   Skip-gram: [one center word]              → dot product with one context → score
#
# CBOW asks:     "given ALL surrounding words, does this center word fit?"
# Skip-gram asks: "given ONE center word, does this context word fit?"
#
# Because skip-gram handles one pair at a time, there is no averaging step.
# That's the only difference in the code — everything else is identical.

class SkipGram(nn.Module):

    def __init__(self, vocab_size, embed_dim):
        # Called ONCE when you write: sg_model = SkipGram(V, embed_dim=8)
        # Builds two empty embedding tables, e.g. vocab_size=12, embed_dim=8:
        #
        #   self.center  → table shape [12 x 8]
        #   self.context → table shape [12 x 8]
        #
        # Both filled with random numbers at first
        super().__init__()
        self.center  = nn.Embedding(vocab_size, embed_dim)
        self.context = nn.Embedding(vocab_size, embed_dim)

    def forward(self, sg_center, sg_context):
        # Called EVERY epoch when you write: sg_model(sg_center, sg_context)
        # PyTorch automatically routes them here via __call__
        #
        # Example values flowing through (embed_dim=4 for simplicity):
        #
        # sg_center  = [2]    ← center id  "small"
        # sg_context = [0]    ← context id "cat"

        c  = self.center(sg_center)
        # Looks up center word vector:
        #   small → [0.3, 0.2, 0.4, 0.1]
        #   c shape: (1, 4)

        ct = self.context(sg_context)
        # Looks up context word vector:
        #   cat → [0.5, 0.1, 0.3, 0.6]
        #   ct shape: (1, 4)

        return (c * ct).sum(dim=1)
        # Element-wise multiply then sum = dot product:
        #   c       = [0.3, 0.2, 0.4, 0.1]
        #   ct      = [0.5, 0.1, 0.3, 0.6]
        #   c * ct  = [0.15, 0.02, 0.12, 0.06]
        #   sum     = 0.35  ← final score for this pair
        #
        # Note: NO averaging here — one center vs one context at a time
        #       This is the key difference from CBOW

In [6]:
# =============================================
# Define vocabulary size and embedding dimension
V = len(vocab)
embed_dim = 8
# =============================================
# V = how many unique words we have (size of our vocabulary)
#     This determines how many rows the embedding table has — one row per word.
#
# embed_dim = how many numbers represent each word (the vector length)
#     embed_dim=8 means each word is described by 8 numbers.
#     Bigger = more expressive but needs more data to train well.
#     Real-world models use 100–300 dimensions; we use 8 because our corpus is tiny.

In [7]:
# ── CBOW context lists have variable length — pad to same size for batching ────
# PyTorch tensors must be rectangular (same length in every row).
# But CBOW context lists have variable length depending on sentence position:
#
#   cat   (position 0) → context = [chases, small]           ← only 2 words
#   chases(position 1) → context = [cat, small, white]       ← 3 words
#   small (position 2) → context = [cat, chases, white, mouse] ← 4 words
#
# We can't stack these into a tensor directly:
#   [[3, 4],          ]   ← length 2
#   [[0, 4, 7],       ]   ← length 3   ✗ not rectangular, tensor will fail
#   [[0, 3, 7, 10],   ]   ← length 4
#
# Solution: pad shorter lists with 0s to match the longest one

# ── Step 1: Find the longest context list ─────────────────────────────────────
max_ctx = max(len(p[0]) for p in cbow_pairs)
# p[0] is the context list for each pair
# len(p[0]) gives its length
# max(...) finds the longest one across all pairs
# e.g. max_ctx = 4  (the middle words have 4 context words)

# ── Step 2: Pad all context lists to length max_ctx ───────────────────────────
cb_context = torch.tensor([p[0] + [0] * (max_ctx - len(p[0])) for p in cbow_pairs])
# For each pair, p[0] is the context list:
#
#   p[0] = [3, 4]           length=2, max_ctx=4, needs 2 zeros
#   [0] * (4 - 2) = [0, 0]
#   [3, 4] + [0, 0] = [3, 4, 0, 0]  ← padded to length 4
#
#   p[0] = [0, 4, 7]        length=3, max_ctx=4, needs 1 zero
#   [0] * (4 - 3) = [0]
#   [0, 4, 7] + [0] = [0, 4, 7, 0]  ← padded to length 4
#
#   p[0] = [0, 3, 7, 10]    length=4, max_ctx=4, needs 0 zeros
#   [0] * (4 - 4) = []
#   [0, 3, 7, 10] + [] = [0, 3, 7, 10]  ← already full length
#
# Final tensor (rectangular, all rows length 4):
#   tensor([[ 3,  4,  0,  0],   ← cat:    [chases, small,  pad,   pad  ]
#           [ 0,  4,  7,  0],   ← chases: [cat,    small,  white, pad  ]
#           [ 0,  3,  7, 10],   ← small:  [cat,    chases, white, mouse]
#           [ 3,  4, 10,  0],   ← white:  [chases, small,  mouse, pad  ]
#           [ 4,  7,  0,  0]])  ← mouse:  [small,  white,  pad,   pad  ]

# ── Step 3: Collect all center word ids ───────────────────────────────────────
cb_center = torch.tensor([p[1] for p in cbow_pairs])
# p[1] is just a single integer (the center word id)
# No padding needed — always exactly one center word per pair
#
#   tensor([0, 3, 4, 7, 10, ...])
#           ↑  ↑  ↑  ↑   ↑
#          cat chases small white mouse



# ── 5a. Train CBOW ────────────────────────────────────────────────────────────
#
# What does "training" mean here?
#   We show the model real pairs and fake pairs over and over.
#   Each time, we measure how wrong the model is (the "loss"),
#   then nudge the embedding vectors slightly to reduce that wrongness.
#   After enough rounds (epochs), the vectors settle into positions where
#   similar words have similar vectors.
#
# The training uses "negative sampling":
#   - Positive sample: a REAL pair from the corpus (context + true center)
#   - Negative sample: a FAKE pair we create (context + random wrong center)
#   The model must learn to score real pairs HIGH and fake pairs LOW.
print("\n--- Training CBOW ---")

# Create a fresh CBOW model with random embeddings
cbow_model = CBOW(V, embed_dim=embed_dim)

# Adam is an optimizer — it decides HOW to update the embedding numbers
# after each round, using the gradients (direction of improvement).
# lr=0.01 is the "learning rate" — how big each update step is.
# Too big → overshoots and never converges. Too small → learns too slowly.
cbow_optim = optim.Adam(cbow_model.parameters(), lr=0.01)

# BCEWithLogitsLoss = Binary Cross-Entropy loss (with built-in sigmoid)
# It measures: "how far is the model's score from the target label?"
#   target=1 (real pair)  → penalise low scores
#   target=0 (fake pair)  → penalise high scores
criterion  = nn.BCEWithLogitsLoss()

# An "epoch" = one complete pass through ALL training pairs.
# 1000 epochs means we show the model every pair 1000 times.
for epoch in range(1000):
    # step 1: positive pairs (real context + real center), and calculate positive loss
    pos_scores = cbow_model(cb_context, cb_center)
    pos_loss   = criterion(pos_scores, torch.ones(len(cbow_pairs)))
    
    # step 2: calculate negative center ids, and prepare negative pairs (real context + fake center), and calculate negative loss
    # Naive version — just random centers (may accidentally pick the true center, but usually wrong)
    # neg_center = torch.randint(0, V, (len(cbow_pairs),))

    # Strict version — ensure fake center is never the true center
    neg_center = []
    for true_center in cb_center.tolist():
        while True:
            fake = torch.randint(0, V, (1,)).item()
            if fake != true_center:       # keep sampling until different
                neg_center.append(fake)
                break
    neg_center = torch.tensor(neg_center)
    neg_scores = cbow_model(cb_context, neg_center)
    neg_loss   = criterion(neg_scores, torch.zeros(len(cbow_pairs)))
    
    # step 3: combine positive and negative loss
    loss = pos_loss + neg_loss
    # step 4: backprop and update model parameters
    # zero_grad() clears old gradients from previous step (otherwise they would accumulate)
    cbow_optim.zero_grad()
    # backward() computes the gradients of the loss with respect to all model parameters (embeddings) and stores them in .grad attributes
    loss.backward()
    # step() updates the model parameters using the computed gradients and the optimization algorithm (Adam in this case)
    cbow_optim.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1} | Loss: {loss.item():.4f}")

# ── 5b. Train Skip-gram ───────────────────────────────────────────────────────
#
# The training process is almost identical to CBOW above.
# The only differences:
#   1. We use the SkipGram model instead of CBOW
#   2. We create negative CONTEXT words (not negative center words)
#      because skip-gram predicts context from center (opposite direction)
#   3. No padding needed — every pair is (one integer, one integer)
print("\n--- Training Skip-gram ---")
sg_model   = SkipGram(V, embed_dim=8)
sg_optim   = optim.Adam(sg_model.parameters(), lr=0.01)

# Skip-gram pairs are always (one integer, one integer) — no padding needed
# because each center word is already exploded into individual pairs:
#   (small, cat), (small, chases), (small, white), (small, mouse)
# Every pair is exactly the same shape → tensor is already rectangular
sg_center  = torch.tensor([p[0] for p in skipgram_pairs])
sg_context = torch.tensor([p[1] for p in skipgram_pairs])

for epoch in range(1000):
    # step 1: positive pairs (real center + real context), and calculate positive loss
    pos_scores = sg_model(sg_center, sg_context)
    pos_loss   = criterion(pos_scores, torch.ones(len(skipgram_pairs)))

    # step 2: calculate negative context ids, and prepare negative pairs (real center + fake context), and calculate negative loss
    # Naive version — just random contexts (may accidentally pick the true context, but usually wrong)
    # neg_context = torch.randint(0, V, (len(skipgram_pairs),))

    # Strict version for skip-gram — ensure fake context is never the true context
    neg_context = []
    for true_context in sg_context.tolist():
        while True:
            fake = torch.randint(0, V, (1,)).item()
            if fake != true_context:       # keep sampling until different
                neg_context.append(fake)
                break
    neg_context = torch.tensor(neg_context)
    
    # Prepare negative pairs (real center + fake context), and calculate negative loss
    neg_scores  = sg_model(sg_center, neg_context)
    neg_loss    = criterion(neg_scores, torch.zeros(len(skipgram_pairs)))

    # step 3: combine positive and negative loss
    loss = pos_loss + neg_loss

    # step 4: backprop and update model parameters
    # zero_grad() clears old gradients from previous step (otherwise they would accumulate)
    sg_optim.zero_grad()
    # backward() computes the gradients of the loss with respect to all model parameters (embeddings) and stores them in .grad attributes
    loss.backward()
    # step() updates the model parameters using the computed gradients and the optimization algorithm (Adam in this case)
    sg_optim.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1} | Loss: {loss.item():.4f}")


# ── Why we need TWO losses ────────────────────────────────────────────────────
# The model needs to learn TWO things simultaneously:
#   1. Score REAL pairs HIGH  → positive loss
#   2. Score FAKE pairs LOW   → negative loss
#
# Without negative loss, the model cheats by outputting high scores for
# EVERYTHING — real and fake pairs. It never learns to discriminate.

# ── Positive loss ─────────────────────────────────────────────────────────────
# Real pairs from corpus — context words that ACTUALLY appear near center word
#   cb_context = [[chases, small, white, mouse], ...]  ← real context
#   cb_center  = [cat, ...]                            ← real center
# >>>pos_scores = cbow_model(cb_context, cb_center)
# pos_scores = [0.8, 0.6, 0.7, ...]  ← one score per real pair

# >>>pos_loss = criterion(pos_scores, torch.ones(len(cbow_pairs)))
# torch.ones = [1, 1, 1, ...]  ← label 1 means "YES these belong together"
# BCE asks: "how far is pos_scores from all 1s?"
#   score=0.8, label=1 → small loss   ✓ model correctly scored high
#   score=0.2, label=1 → large loss   ✗ model wrongly scored low

# ── Negative loss ─────────────────────────────────────────────────────────────
# Fake pairs — same context but RANDOM center word (probably wrong)
# >>>neg_center = torch.randint(0, V, (len(cbow_pairs),))
# e.g. neg_center = [7, 2, 11, ...]  ← random word ids (white, bird, fly...)
#      cb_context stays the same    ← still [chases, small, white, mouse]
#
# So fake pair = (context of "cat") + (random center "fly")
# These almost certainly don't belong together

# >>>neg_scores = cbow_model(cb_context, neg_center)
# neg_scores = [0.3, 0.5, 0.2, ...]  ← scores for fake pairs

# >>>neg_loss = criterion(neg_scores, torch.zeros(len(cbow_pairs)))
# torch.zeros = [0, 0, 0, ...]  ← label 0 means "NO these don't belong together"
# BCE asks: "how far is neg_scores from all 0s?"
#   score=0.1, label=0 → small loss   ✓ model correctly scored low
#   score=0.8, label=0 → large loss   ✗ model wrongly scored high

# ── Total loss ────────────────────────────────────────────────────────────────
# >>> loss = pos_loss + neg_loss
# Combined signal pushes the model in BOTH directions at once:
#
#   pos_loss pushes → score REAL pairs higher
#   neg_loss pushes → score FAKE pairs lower
#
# After 500 epochs:
#   (cat,   [chases,small,white,mouse]) → high score  ✓
#   (fly,   [chases,small,white,mouse]) → low score   ✓
#   (dog,   [chases,big,black,mouse])   → high score  ✓
#   (mouse, [chases,big,black,mouse])   → low score   ✓


--- Training CBOW ---
Epoch 100 | Loss: 0.8418
Epoch 200 | Loss: 0.4634
Epoch 300 | Loss: 0.4156
Epoch 400 | Loss: 0.3445
Epoch 500 | Loss: 0.3048
Epoch 600 | Loss: 0.3825
Epoch 700 | Loss: 0.3050
Epoch 800 | Loss: 0.4430
Epoch 900 | Loss: 0.3408
Epoch 1000 | Loss: 0.2655

--- Training Skip-gram ---
Epoch 100 | Loss: 1.2329
Epoch 200 | Loss: 0.8857
Epoch 300 | Loss: 0.8185
Epoch 400 | Loss: 0.8367
Epoch 500 | Loss: 0.7811
Epoch 600 | Loss: 0.8519
Epoch 700 | Loss: 0.7900
Epoch 800 | Loss: 0.6725
Epoch 900 | Loss: 0.7163
Epoch 1000 | Loss: 0.7823


In [8]:
# ── Show sentences again for reference ─────────────────────────────────────────
print(f"Sentences:")
for i in range(len(corpus)):
     print(f"       {corpus[i]}")
     
# ── 6. Compare similarity results ─────────────────────────────────────────────
#
# Now that training is done, let's see if the learned embeddings make sense.
# We check: for each word, which other words ended up with the most similar vectors?
#
# If training worked, we'd expect:
#   - animals (cat, dog, bird) to be similar to each other
#   - sizes (small, big, tiny) to be similar to each other
#   - colours (white, black, blue, ...) to be similar to each other
#   - prey (mouse, fly) to be similar to each other
#
# How most_similar() works step by step:
#   1. Grab the center embedding table from the trained model
#      (remember, we have two tables — center and context — we use center)
#   2. Normalize each word's vector to length 1 (unit vectors)
#      This lets us use dot product as cosine similarity (range -1 to +1)
#   3. Compute dot product between the target word and ALL other words
#      High dot product = vectors point in similar direction = similar meaning
#   4. Set the word's similarity with itself to -1 (so it doesn't show up as #1)
#   5. Return the top_k most similar words with their similarity scores

def most_similar(model, word, top_k=3):
    E    = model.center.weight.data          # shape: [V, embed_dim] — all word vectors
    E    = E / E.norm(dim=1, keepdim=True)   # normalize to unit length (for cosine similarity)
    sims = E @ E[vocab[word]]                # dot product of target word vs all words → [V] scores
    sims[vocab[word]] = -1                   # exclude the word itself
    top  = sims.topk(top_k).indices.tolist() # indices of the top_k highest scores
    return [(idx2w[i], round(sims[i].item(), 3)) for i in top]



# ── CBOW results ──────────────────────────────────────────────────────────────
# The number in parentheses is the cosine similarity score:
#   1.0  = identical direction (perfect match)
#   0.0  = unrelated (perpendicular)
#  -1.0  = opposite direction
print("\n─── CBOW Similarity Results ──────────────────────────────────────────")
print(f"{'Word':<10} {'Top 1':<20} {'Top 2':<20} {'Top 3':<20}")
print("─" * 70)
for word in words:
    results = most_similar(cbow_model, word)
    top1 = f"{results[0][0]} ({results[0][1]})"
    top2 = f"{results[1][0]} ({results[1][1]})"
    top3 = f"{results[2][0]} ({results[2][1]})"
    print(f"{word:<10} {top1:<20} {top2:<20} {top3:<20}")

# ── Skip-gram results ─────────────────────────────────────────────────────────
# Same evaluation but using the skip-gram model's embeddings.
# Results will differ because CBOW and skip-gram learn differently:
#   - CBOW averages context → tends to smooth over details
#   - Skip-gram handles one pair at a time → can capture rarer relationships
# With a corpus this small, both models will be noisy — don't expect perfect results.
print("\n─── Skip-gram Similarity Results ─────────────────────────────────────")
print(f"{'Word':<10} {'Top 1':<20} {'Top 2':<20} {'Top 3':<20}")
print("─" * 70)
for word in words:
    results = most_similar(sg_model, word)
    top1 = f"{results[0][0]} ({results[0][1]})"
    top2 = f"{results[1][0]} ({results[1][1]})"
    top3 = f"{results[2][0]} ({results[2][1]})"
    print(f"{word:<10} {top1:<20} {top2:<20} {top3:<20}")

Sentences:
       money at bank
       cash at bank
       funds at bank
       dollars at bank
       money at vault
       cash at vault
       funds at vault
       dollars at vault
       fish near bank
       river near bank
       trees near bank
       grass near bank
       fish near shore
       river near shore
       trees near shore
       grass near shore

─── CBOW Similarity Results ──────────────────────────────────────────
Word       Top 1                Top 2                Top 3               
──────────────────────────────────────────────────────────────────────
money      dollars (0.927)      funds (0.795)        cash (0.785)        
cash       funds (0.829)        dollars (0.797)      money (0.785)       
funds      dollars (0.881)      cash (0.829)         money (0.795)       
dollars    money (0.927)        funds (0.881)        cash (0.797)        
vault      bank (0.587)         fish (0.439)         dollars (0.349)     
fish       grass (0.953)        trees (0.8

In [10]:
from sklearn.decomposition import PCA
import plotly.express as px

E = cbow_model.center.weight.data.numpy()
coords = PCA(n_components=3).fit_transform(E)
words_all = [idx2w[i] for i in range(len(vocab))]

fig = px.scatter_3d(x=coords[:,0], y=coords[:,1], z=coords[:,2],
                    text=words_all, title="CBOW Embeddings (PCA → 3D)",
                    width=1000, height=800)

fig.update_traces(marker=dict(size=8), textposition="top center")
fig.show()


## Observation: Why Word2Vec Fails on "Bank"

What we just saw in the 3D plot:
- **money, cash, funds, dollars, vault** → one tight cluster
- **fish, river, trees, grass, shore** → another tight cluster
- **"bank"** → stuck in the middle, belonging to neither

**Why?** Word2Vec assigns **one fixed vector per word**. It has no idea
which sentence "bank" came from — it merges both meanings into one point.

## How Contextual Embeddings (BERT, GPT) Solve This

| | Word2Vec | BERT / GPT |
|---|---|---|
| Vector for "bank" | **one** vector, always the same | **different** vector per sentence |
| "deposit money at **bank**" | → same point | → near money cluster |
| "fish swim near **bank**" | → same point | → near river cluster |

The key difference: contextual models read the **entire sentence** before
producing a vector. So the same word gets a different representation
depending on its neighbours.


In [12]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

def get_word_embedding(sentence, target_word):
    """Get BERT's contextual embedding for a specific word in a sentence."""
    inputs = tokenizer(sentence, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    with torch.no_grad():
        outputs = model(**inputs)
    # Find the target word's position and return its vector
    for i, token in enumerate(tokens):
        if token == target_word:
            return outputs.last_hidden_state[0, i]

# Same word "bank" — two different sentences
money_bank = get_word_embedding("deposit money at bank", "bank")
river_bank = get_word_embedding("fish swim near bank",   "bank")




Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18930.97it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
# How similar are they? (cosine similarity)
cos_sim = torch.nn.functional.cosine_similarity(money_bank, river_bank, dim=0)
print(f'BERT similarity between "bank" (money) and "bank" (river): {cos_sim:.3f}')

BERT similarity between "bank" (money) and "bank" (river): 0.621


```
══════════════════════════════════════════════════════════════
  Word2Vec  vs  Contextual Embeddings (BERT / GPT / modern LLMs)
══════════════════════════════════════════════════════════════
```

WORD2VEC — the table IS the model

──────────────────────────────────

TRAINING:  Adjust the TABLE so "bank" row is near "money" row and far from "fish"



```
       ┌───────────────────────────┐
       │  Embedding Table          │  ← this is the ONLY thing that gets trained
       │                           │
       │  bank  → [0.3, 0.1, 0.7]  │
       │  money → [0.5, 0.2, 0.6]  │
       │  fish  → [0.1, 0.8, 0.3]  │
       └───────────────────────────┘
```



INFERENCE: Just look up the row. Done.

```

"deposit money at bank"  →  table["bank"]  →  [0.3, 0.1, 0.7]
"fish swim near bank"    →  table["bank"]  →  [0.3, 0.1, 0.7]
↑ SAME vector!

```

No context at inference time. Context only influenced training.


──────────────────────────────────────────────────────────────


BERT / GPT — embedding + transformer (modern LLM view)

──────────────────────────────────────────────────────

TRAINING:  Adjust EVERYTHING end-to-end



```
       ┌──────────────────────────────┐
       │  Tokenizer (fixed)           │  ← NOT trained with the model
       │  "playing" → ["play","##ing"]│
       └──────────────┬───────────────┘
                      ↓
       ┌──────────────────────────────┐
       │  Embedding Table             │  ← randomly initialized, then trained
       │  (learned lookup)            │
       └──────────────┬───────────────┘
                      ↓
       ┌──────────────────────────────┐
       │ Positional Encoding          │  ← added to embeddings
       └──────────────┬───────────────┘
                      ↓
       ┌──────────────────────────────────────────────────┐
       │   TRANSFORMER (many layers)                      │
       │                                                  │
       │   This is the MAIN model (not just an add-on)    │
       │                                                  │
       │   Each layer:                                    │
       │     - Self-attention (mix tokens)                │
       │     - Feedforward (transform features)           │
       │                                                  │
       │   All weights are trained together with          │
       │   the embedding table via backpropagation.       │
       │                                                  │
       │   Layers progressively refine meaning:           │
       │     Layer 1: rough interaction                   │
       │     Layer 5: clearer relationships               │
       │     Layer N: rich contextual meaning             │
       │                                                  │
       └──────────────┬───────────────────────────────────┘
                      ↓
              contextual embeddings
```



INFERENCE: Same pipeline, but no weight updates

```

"deposit money at bank"

Step 1: tokenization
Step 2: embedding lookup  (same starting vector as always)
Step 3: transformer layers (THIS is where meaning happens)

Result:
"bank" → money-related vector

```

```

"fish swim near bank"

Step 1: SAME embedding lookup
Step 2: DIFFERENT context processing

Result:
"bank" → river-related vector

```

---

IMPORTANT CORRECTIONS FOR MODERN LLMs

─────────────────────────────────────

1. The embedding table is NOT the main model  
   → It is just the **input layer**

2. The transformer is NOT “one extra step”  
   → It is **the core of the model**

3. Embeddings are usually:
   → randomly initialized  
   → trained jointly with the transformer  
   → NOT pretrained like Word2Vec in most modern systems

4. Meaning is NOT stored in the embedding table  
   → It is **computed dynamically by the transformer**

---

THE REAL DIFFERENCE

───────────────────

Word2Vec:

```
output = table[word]

```

Transformer (BERT/GPT):

```
output = Transformer( embeddings of ALL tokens in the sequence )

```



THE KEY INSIGHT

────────────────

Word2Vec:
→ “What does this word generally mean?”

Transformer:
→ “What does this word mean in THIS sentence?”



FINAL MENTAL MODEL

──────────────────

Word2Vec:
  word → fixed meaning

Transformer:
  word → initial vector → repeatedly updated by context → final meaning

